In [1]:
#1. Install the ultralytics package
#!pip install ultralytics
!pip install -U ultralytics

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 1.7 MB/s eta 0:00:01
   -------------------------- ------------- 0.8/1.2 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.5 MB/s  0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.7
    Uninstalling ultralytics-8.4.7:
      Successfully uninstalled ultralytics-8.4.7



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#2. Import necessary libraries
# Install openpyxl for Excel file handling
# This is necessary for saving metrics to Excel files
!pip install openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

# 3. สร้างไฟล์ data.yaml สำหรับการ train YOLO
data_yaml = """
# ❗ เปลี่ยน path ให้เป็น path จริงบนเครื่องของคุณ
path: C:/train_5_2_69/dataset

train: images/train
val: images/val
test: images/test

nc: 12

names:
  - Alovera
  - cucumber
  - Galangal
  - Garlic
  - horapa
  - Houttuynia_cordata
  - Ivy_Gourd
  - khaproa
  - Mangosteen_Peel
  - pluLeaf
  - Snake_Plant
  - Turmeric

"""

with open("data.yaml", "w", encoding="utf-8") as f:
        f.write(data_yaml)

In [4]:
# 4.เช็กและลบ label ผิดก่อน train:
import os

def clean_labels(image_dir, label_dir):
    removed = 0
    # Check if label_dir exists before listing its contents
    if not os.path.exists(label_dir):
        print(f"Warning: Label directory not found at {label_dir}. Skipping cleaning for this set.")
        return

    for file in os.listdir(label_dir):
        label_path = os.path.join(label_dir, file)
        # Assuming images are .jpg or .png
        image_path_jpg = os.path.join(image_dir, file.replace('.txt', '.jpg'))
        image_path_png = os.path.join(image_dir, file.replace('.txt', '.png'))

        if not os.path.exists(image_path_jpg) and not os.path.exists(image_path_png):
            try:
                os.remove(label_path)
                removed += 1
            except OSError as e:
                print(f"Error deleting file {label_path}: {e}")

    print(f"✅ ลบ label ที่ไม่มีภาพออกแล้ว: {removed} ไฟล์ ใน {label_dir}")

# เรียกใช้กับทุกชุด โดยใช้พาธที่ถูกต้อง
base_dir = r'C:\train20_12_68'
clean_labels(os.path.join(base_dir, 'images', 'train'), os.path.join(base_dir, 'labels', 'train'))
clean_labels(os.path.join(base_dir, 'images', 'val'), os.path.join(base_dir, 'labels', 'val'))
clean_labels(os.path.join(base_dir, 'images', 'test'), os.path.join(base_dir, 'labels', 'test'))

In [6]:
# 5.อ่านและแสดงเนื้อหาของไฟล์ data.yaml
# ไฟล์นี้ใช้กำหนดการตั้งค่า dataset สำหรับการฝึกโมเดล YOLO

yaml_path = r'C:\train_5_2_69\data.yaml'
try:
    with open(yaml_path, 'r', encoding='utf-8') as f:
        yaml_content = f.read()
    print("Content of data.yaml:")
    print(yaml_content)
except FileNotFoundError:
    print(f"Error: {yaml_path} not found. Please ensure the data.yaml file exists in the specified location.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

Content of data.yaml:

# ❗ เปลี่ยน path ให้เป็น path จริงบนเครื่องของคุณ
path: C:/train_5_2_69/dataset

train: images/train
val: images/val
test: images/test

nc: 12

names:
  - Alovera
  - cucumber
  - Galangal
  - Garlic
  - horapa
  - Houttuynia_cordata
  - Ivy_Gourd
  - khaproa
  - Mangosteen_Peel
  - pluLeaf
  - Snake_Plant
  - Turmeric




In [6]:
# 6. Train YOLOv8 Model and visualize results
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import os
import torch

        # 1. Train YOLOv8
        model = YOLO('yolov8s.pt')  # เปลี่ยนเป็น yolov9n.pt, ...
        results = model.train(
    data=r"C:/train20_12_68/data.yaml",
    epochs=100,
    imgsz=960,
    batch=8,            # ถ้า OOM ลดเป็น 4
    device=0,
    optimizer="AdamW",

    augment=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.35,
    degrees=5,
    translate=0.08,
    scale=0.4,
    shear=1.5,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,

    mosaic=0.3,         # dataset ใหญ่แล้ว ลดให้พอดี
    mixup=0.0
)

        print("✅ การฝึกโมเดลเสร็จสมบูรณ์แล้ว!")
        
        # 2. Auto-find results/metrics files under runs/detect and convert CSV->XLSX when needed
        runs_base = os.path.abspath('runs')
        patterns = [
            os.path.join(runs_base, 'detect', '**', 'results.xlsx'),
            os.path.join(runs_base, 'detect', '**', 'results.csv'),
            os.path.join(runs_base, 'detect', '**', 'metrics.csv'),
            os.path.join(runs_base, 'detect', '**', 'metrics.xlsx'),
        ]
        candidates = []
        for p in patterns:
            candidates.extend(glob.glob(p, recursive=True))
        
        if not candidates:
            print('Error: ไม่พบไฟล์ metrics/results ภายใต้ runs\\detect/')
            # แสดงโฟลเดอร์ที่มีอยู่เพื่อดีบั๊ก
            detect_dirs = glob.glob(os.path.join(runs_base, 'detect', '*')) if os.path.exists(os.path.join(runs_base, 'detect')) else []
            print('folders under runs\\detect:', sorted(detect_dirs))
        else:
            latest = max(candidates, key=os.path.getmtime)
            print('พบไฟล์:', latest)
            # ถ้าเป็น CSV ให้แปลงเป็น Excel (ปลอดภัยต่อการอ่านแถวที่ไม่ตรงรูปแบบ)
            if latest.lower().endswith('.csv'):
                excel_path = latest.rsplit('.', 1)[0] + '.xlsx'
                try:
                    df = pd.read_csv(latest)
                except Exception:
                    try:
                        df = pd.read_csv(latest, engine='python', on_bad_lines='skip')
                    except Exception as e:
                        print('ไม่สามารถอ่าน CSV ได้:', e)
                        df = None
                if df is not None:
                    try:
                        df.to_excel(excel_path, index=False)
                        print('✅ แปลง CSV -> Excel:', excel_path)
                    except Exception as e:
                        print('แปลงเป็น Excel ล้มเหลว:', e)
            else:
                excel_path = latest
                try:
                    df = pd.read_excel(excel_path)
                except Exception as e:
                    print('อ่าน Excel ล้มเหลว:', e)
                    df = None
        
            if df is None:
                print('ไม่สามารถโหลดตาราง metrics ได้ จากไฟล์:', latest)
            else:
                print('Columns:', df.columns.tolist())
        
                metrics_to_plot = {
                    'metrics/mAP_0.5:0.95': 'mAP50-95 vs. Epoch',
                    'metrics/mAP_0.5': 'mAP50 vs. Epoch',
                    'metrics/precision': 'Precision vs. Epoch',
                    'metrics/recall': 'Recall vs. Epoch'
                }
        
                # Normal plotting loop
                for metric_col, title in metrics_to_plot.items():
                    if metric_col in df.columns:
                        plt.figure(figsize=(10, 6))
                        plt.plot(df.index, df[metric_col])
                        plt.xlabel('Epoch')
                        plt.ylabel(metric_col.split('/')[-1])
                        plt.title(title)
                        plt.grid(True)
                        plt.show()
                    else:
                        # try fuzzy matches like columns that contain 'mAP' or 'precision'
                        matches = [c for c in df.columns.tolist() if metric_col.split('/')[-1] in c or metric_col.split('/')[-1].replace('_', '') in c.replace('_', '')]
                        if matches:
                            m = matches[0]
                            print(f"Found similar column '{m}' for requested '{metric_col}', plotting it.")
                            plt.figure(figsize=(10,6))
                            plt.plot(df.index, df[m])
                            plt.xlabel('Epoch')
                            plt.ylabel(m)
                            plt.title(title)
                            plt.grid(True)
                            plt.show()
                        else:
                            print(f"Warning: Metric column '{metric_col}' not found in {excel_path}.")

IndentationError: unexpected indent (4257601982.py, line 9)

In [3]:
# 7.Evaluate the model on the test set and save metrics to Excel and show confusion matrix
#
import pandas as pd
import numpy as np
from ultralytics import YOLO
import torch
import os
from PIL import Image
import matplotlib.pyplot as plt

# 1. Evaluate the model on the test set
model = YOLO(r'C:\train_5_2_69\runs\detect\yolov8s(lre-3)(BG)\train4\weights\best.pt')  # ปรับ path ให้ตรงกับ best.pt ล่าสุด
metrics = model.val(split='test', data=r'C:\train_5_2_69\data.yaml')
print(metrics)

# 2. Extract relevant metrics
metrics_data = {
    'Metric': ['mAP50-95', 'mAP50', 'Precision', 'Recall'],
    'Overall': [
        metrics.box.maps.mean() if hasattr(metrics.box, 'maps') else np.nan,
        metrics.box.map50 if hasattr(metrics.box, 'map50') else np.nan,
        metrics.box.mp if hasattr(metrics.box, 'mp') else np.nan,
        metrics.box.mr if hasattr(metrics.box, 'mr') else np.nan
    ]
}

# Per-class metrics
num_classes = len(metrics.names)
ap_per_class = metrics.box.ap if hasattr(metrics.box, 'ap') and isinstance(metrics.box.ap, np.ndarray) and len(metrics.box.ap) == num_classes else [np.nan] * num_classes
ap50_per_class = metrics.box.ap50 if hasattr(metrics.box, 'ap50') and isinstance(metrics.box.ap50, np.ndarray) and len(metrics.box.ap50) == num_classes else [np.nan] * num_classes
p_per_class = metrics.box.p if hasattr(metrics.box, 'p') and isinstance(metrics.box.p, np.ndarray) and len(metrics.box.p) == num_classes else [np.nan] * num_classes
r_per_class = metrics.box.r if hasattr(metrics.box, 'r') and isinstance(metrics.box.r, np.ndarray) and len(metrics.box.r) == num_classes else [np.nan] * num_classes

for i, class_name in enumerate(metrics.names):
    metrics_data[class_name] = [ap_per_class[i], ap50_per_class[i], p_per_class[i], r_per_class[i]]

# 3. Save metrics to Excel
metrics_df = pd.DataFrame(metrics_data)
excel_output_path = r'C:\train_5_2_69\runs\detect\yolov8s(lre-3)(BG)\evaluation_metrics.xlsx'
metrics_df.to_excel(excel_output_path, index=False)
print(f"✅ Evaluation metrics saved to {excel_output_path}")

# 4. Show confusion matrix
conf_matrix_path = r'C:\train_5_2_69\runs\detect\yolov8s(lre-3)(BG)\val\confusion_matrix.png'
if os.path.exists(conf_matrix_path):
    img = Image.open(conf_matrix_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title("Confusion Matrix")
    plt.show()
else:
    print(f"ไม่พบไฟล์ {conf_matrix_path}")

# 5. Predict and show results on test images
results = model.predict(
    source=r'C:\train_5_2_69\dataset\images\test',
    conf=0.5,
    save=True  # จะบันทึกภาพที่ตรวจจับแล้วไว้ใน runs/
)

# 6. Save predict log to Excel
log_data = []
for result in results:
    img_path = result.path if hasattr(result, 'path') else None
    for box, conf, cls in zip(result.boxes.xyxy.cpu().numpy(),
                              result.boxes.conf.cpu().numpy(),
                              result.boxes.cls.cpu().numpy()):
        log_data.append({
            'image': img_path,
            'class': result.names[int(cls)],
            'confidence': float(conf),
            'x1': float(box[0]),
            'y1': float(box[1]),
            'x2': float(box[2]),
            'y2': float(box[3])
        })
df_log = pd.DataFrame(log_data)
log_path = r'C:\train_5_2_69\runs\detect\yolov8s(lre-3)(BG)\predict_log.xlsx'
df_log.to_excel(log_path, index=False)
print(f"✅ Predict log saved to {log_path}")

# 7. Show example prediction images
for result in results[:3]:  # แสดงแค่ 3 ภาพตัวอย่าง
    img = result.plot()  # วาดกรอบผลลัพธ์ลงบนภาพ
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Prediction Result")
    plt.show()

Ultralytics 8.4.14  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 73 layers, 11,130,228 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 193.551.5 MB/s, size: 336.8 KB)
val: Scanning C:\train_5_2_69\dataset\labels\test.cache... 2400 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2400/2400  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 150/150 3.7it/s 40.7s0.3ss
                   all       2400       3041      0.974      0.951      0.975      0.944
               Alovera        180        241      0.979       0.95      0.987      0.923
              cucumber        204        211      0.997      0.995      0.995      0.991
              Galangal        202        206      0.998          1      0.995      0.995
                Garlic        184        232      0.998          1      0.995      0.995
                horap

<Figure size 640x480 with 1 Axes>


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2400 C:\train_5_2_69\dataset\images\test\Alovera_0006.png: 640x640 1 Alovera, 38.4ms
image 2/2400 C:\train_5_2_69\dataset\images\test\Alovera_0009.png: 640x640 1 Alovera, 38.3ms
image 3/2400 C:\train_5_2_69\dataset\images\test\Alovera_0019.png: 640x640 1 Alovera, 25.6ms
image 4/2400 C:\train_5_2_69\dataset\images\test\Alovera_0031.png: 640x640 1 Alovera, 24.4ms
image 5/2400 C:\train_5_2_69\dataset\images\test\Alovera_0033.png: 640x640 1 Alovera, 24.

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

In [13]:
#export YOLOv8 เป็น ONNX และ TensorFlow SavedMode
# 8. Export YOLOv8 model to ONNX and TensorFlow SavedModel
# 1. ติดตั้ง tensorflowjs
#!pip install tensorflowjs
## 2. คำสั่งนี้ใน terminal: tensorflowjs_converter --input_format=tf_saved_model runs\detect\train\weights\best_saved_model ./best_tfjs

from ultralytics import YOLO

# โหลดโมเดลที่ train เสร็จแล้ว
model = YOLO(r'C:\train20_12_68\results_yolov8s(lr1e-4)(1)\train\weights\best.pt')

# Export เป็น ONNX
model.export(format='onnx', dynamic=True, opset=12)
# จะได้ไฟล์ runs\detect\train\weights\best.onnx

# Export เป็น TensorFlow SavedModel
model.export(format='tf')
# จะได้โฟลเดอร์ runs\detect\train\weights\best_saved_model/


Ultralytics 8.3.241  Python-3.11.9 torch-2.6.0+cu124 CPU (AMD Ryzen 7 7435HS)
Model summary (fused): 72 layers, 11,130,228 parameters, 0 gradients, 28.5 GFLOPs

PyTorch: starting from 'C:\train20_12_68\results_yolov8s(lr1e-4)(1)\train\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 16, 8400) (21.5 MB)

ONNX: starting export with onnx 1.19.0 opset 12...
ONNX: slimming with onnxslim 0.1.82...
ONNX: export success  4.7s, saved as 'C:\train20_12_68\results_yolov8s(lr1e-4)(1)\train\weights\best.onnx' (43.0 MB)

Export complete (5.8s)
Results saved to C:\train20_12_68\results_yolov8s(lr1e-4)(1)\train\weights
Predict:         yolo predict task=detect model=C:\train20_12_68\results_yolov8s(lr1e-4)(1)\train\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\train20_12_68\results_yolov8s(lr1e-4)(1)\train\weights\best.onnx imgsz=640 data=C:/train20_12_68/data.yaml  
Visualize:       https://netron.app
WARNING Invalid export format='tf', up

ModuleNotFoundError: No module named 'onnx2tf'

In [ ]:
# โดดข้ามเลยจ้า เครื่องไม่แรงพอแน่ๆ
# 8. Compare YOLO versions and visualize results
#
#
#
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import os

model_versions = [
    {'name': 'YOLOv8', 'weight': 'yolov8n.pt'},
    {'name': 'YOLOv9', 'weight': 'yolov9n.pt'},
    {'name': 'YOLOv10', 'weight': 'yolov10n.pt'},
    {'name': 'YOLOv11', 'weight': 'yolov11n.pt'},
]

results_summary = []

for model_info in model_versions:
    print(f"=== Training {model_info['name']} ===")
    model = YOLO(model_info['weight'])
    results = model.train(
        data=r'C:\Users\Wongpanya.Nu\DB_BAN\data.yaml',
        epochs=1,
        imgsz=640,
        batch=16,
        device='cuda:0',
        optimizer='AdamW',
        project=f'results_{model_info["name"]}',  # แยกโฟลเดอร์
        name='exp'
    )
    # Evaluate
    metrics = model.val(split='test', data=r'C:\Users\Wongpanya.Nu\DB_BAN\data.yaml')
    # เก็บ summary
    results_summary.append({
        'Model': model_info['name'],
        'mAP50-95': metrics.box.maps.mean() if hasattr(metrics.box, 'maps') else None,
        'mAP50': metrics.box.map50 if hasattr(metrics.box, 'map50') else None,
        'Precision': metrics.box.mp if hasattr(metrics.box, 'mp') else None,
        'Recall': metrics.box.mr if hasattr(metrics.box, 'mr') else None
    })
    # สามารถ save confusion matrix, log, ฯลฯ เพิ่มเติมได้

# สร้าง DataFrame เปรียบเทียบ
df_compare = pd.DataFrame(results_summary)
print(df_compare)

# Plot เปรียบเทียบ
plt.figure(figsize=(10,6))
plt.bar(df_compare['Model'], df_compare['mAP50-95'])
plt.ylabel('mAP50-95')
plt.title('Comparison of mAP50-95 between YOLO versions')
plt.show()

# Export summary to Excel
df_compare.to_excel('compare_yolo_versions.xlsx', index=False)

In [ ]:
# Structure of results directory
""" DB_BAN/
│
├─ results_yolov8/
│    ├─ train/...
│    ├─ val/...
│    ├─ metrics.csv
│    ├─ predict_log.xlsx
│    └─ confusion_matrix.png
├─ results_yolov9/
│    ├─ train/...
│    ├─ val/...
│    ├─ metrics.csv
│    ├─ predict_log.xlsx
│    └─ confusion_matrix.png
...
"

In [ ]:
"""
โปรดอ่านกให้เข้าใจก่อนเริ่มใช้งาน
# 1. ติดตั้ง ultralytics และ openpyxl  
# 2. สร้างไฟล์ data.yaml สำหรับการ train YOLO
# 3. เช็กและลบ label ผิดก่อน train
# 4. อ่านและแสดงเนื้อหาของไฟล์ data.yaml
# 5. Train YOLOv8 Model and visualize results
# 6. Evaluate the model on the test set and save metrics to Excel and show confusion matrix
# 7. Predict and show results on test images
# 8. Compare YOLO versions and visualize results  โดดข้ามเลนช้าเครื่องช้า ให้ไปทำ 9-10
# 9. Export YOLOv8-v11 ทีละครั้ง model 
# 10. Export model to ONNX and TensorFlow SavedModel
"""

In [8]:
from collections import Counter
import glob

cls_counter = Counter()

for f in glob.glob(r"C:/train20_12_68/labels/*/*.txt"):
    with open(f) as file:
        for line in file:
            cls = int(line.split()[0])
            cls_counter[cls] += 1

print(cls_counter)


Counter({3: 1536, 8: 960, 11: 851, 4: 837, 1: 833, 5: 823, 0: 821, 10: 821, 2: 816, 7: 814, 6: 807, 9: 793})


In [6]:
import torch, torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda:", torch.version.cuda)


ModuleNotFoundError: No module named 'torch'

In [1]:
# =========================================================
# Train / Evaluate / Predict / Log (YOLOv8) - FULL VERSION
# =========================================================
from ultralytics import YOLO
import pandas as pd
import numpy as np
import os
import shutil
from datetime import datetime

# =========================
# CONFIG
# =========================
model_name = "yolov8s.pt"
result_dir = "yolov8s(lre-3)(BG)"
data_yaml = r"C:/train_5_2_69/data.yaml"

os.makedirs(result_dir, exist_ok=True)

# =========================
# 1) TRAIN
# =========================
model = YOLO(model_name)

train_results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,

    batch=8,              # ✅ ลดจาก 16 -> 8 (ถ้ายัง OOM ค่อยลดเป็น 4)
    device=0,
    optimizer="AdamW",

    lr0=3e-4,             # ✅ ให้พอเรียนรู้ (1e-4 ช้าเกิน)
    lrf=1e-4,
    warmup_epochs=3,

    augment=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.35,
    degrees=5,
    translate=0.08,
    scale=0.4,
    shear=1.5,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.2,           # ✅ ลดนิด ลด RAM/VRAM + ลดความแกว่ง
    mixup=0.0,

    project=result_dir,
    name="train",

    workers=2,            # ✅ ลด worker กันกิน RAM แล้วค้าง/รีสตาร์ท
    cache=False,
    amp=True,             # ✅ เปิดไว้ช่วยลด VRAM
    close_mosaic=10       # ✅ ช่วงท้ายปิด mosaic ให้เสถียร + เบาขึ้น
)


# =========================
# 2) VALIDATION (TEST)
# =========================
metrics = model.val(
    data=data_yaml,
    split="test",
    project=result_dir,
    name="val"
)

# =========================
# 3) METRICS SUMMARY
# =========================
metrics_df = pd.DataFrame({
    "Metric": ["mAP50-95", "mAP50", "Precision", "Recall"],
    "Overall": [
        metrics.box.maps.mean(),
        metrics.box.map50,
        metrics.box.mp,
        metrics.box.mr,
    ],
})

metrics_csv = os.path.join(result_dir, "metrics.csv")
metrics_xlsx = os.path.join(result_dir, "evaluation_metrics.xlsx")

metrics_df.to_csv(metrics_csv, index=False)
metrics_df.to_excel(metrics_xlsx, index=False)

# =========================
# 4) PREDICT + LOG
# =========================
predict_results = model.predict(
    source=r"C:/train_5_2_69/dataset/images/test",
    conf=0.5,
    save=True,
    project=result_dir,
    name="predict"
)

log_rows = []
for r in predict_results:
    img_path = r.path
    for box, conf, cls in zip(
        r.boxes.xyxy.cpu().numpy(),
        r.boxes.conf.cpu().numpy(),
        r.boxes.cls.cpu().numpy(),
    ):
        log_rows.append({
            "image": img_path,
            "class": r.names[int(cls)],
            "confidence": float(conf),
            "x1": float(box[0]),
            "y1": float(box[1]),
            "x2": float(box[2]),
            "y2": float(box[3]),
        })

predict_log = pd.DataFrame(log_rows)
predict_log.to_excel(
    os.path.join(result_dir, "predict_log.xlsx"),
    index=False
)

# =========================
# 5) EXTRA STRUCTURE / COPY
# =========================
train_dir = os.path.join(result_dir, "train")
val_dir = os.path.join(result_dir, "val")

# Copy key plots from val
for fn in [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
]:
    src = os.path.join(val_dir, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(result_dir, fn))

# Copy train results
for fn in ["results.png", "results.csv"]:
    src = os.path.join(train_dir, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(result_dir, f"train_{fn}"))

# Copy best / last weights
for w in ["best.pt", "last.pt"]:
    src = os.path.join(train_dir, "weights", w)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(result_dir, w))

# =========================
# 6) SAVE LR + README
# =========================
with open(os.path.join(result_dir, "ค่า lr.txt"), "w", encoding="utf-8") as f:
    f.write(f"run: {result_dir}\n")
    f.write(f"time: {datetime.now()}\n\n")
    f.write("model: yolov8s\n")
    f.write("lr0: 1e-4\n")
    f.write("lrf: 1e-5\n")
    f.write("warmup_epochs: 3\n")
    f.write("mosaic: 0.3\n")
    f.write("mixup: 0.0\n")

with open(os.path.join(result_dir, "README.txt"), "w", encoding="utf-8") as f:
    f.write("RESULT STRUCTURE\n\n")
    f.write("train/   : training outputs & weights\n")
    f.write("val/     : validation results\n")
    f.write("predict/ : prediction images\n\n")
    f.write("metrics.csv / evaluation_metrics.xlsx : summary metrics\n")
    f.write("predict_log.xlsx : per-image detection log\n")
    f.write("best.pt : best trained model\n")

print("✅ Training pipeline completed successfully")

Ultralytics 8.4.14  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/train_5_2_69/data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.0001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.2, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patien

In [1]:
import torch, torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda:", torch.version.cuda)


ModuleNotFoundError: No module named 'torch.hub'

In [1]:
import ultralytics
print(ultralytics.__version__)

from ultralytics import YOLO
print(YOLO)


8.4.6
<class 'ultralytics.models.yolo.model.YOLO'>


In [14]:
import ultralytics
print(ultralytics)
print(ultralytics.__file__)
print(dir(ultralytics))


<module 'ultralytics' from 'c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\ultralytics\\__init__.py'>
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\__init__.py
['ASSETS', 'FastSAM', 'MODELS', 'NAS', 'RTDETR', 'SAM', 'SETTINGS', 'TYPE_CHECKING', 'YOLO', 'YOLOE', 'YOLOWorld', '__all__', '__builtins__', '__cached__', '__dir__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'cfg', 'checks', 'data', 'download', 'engine', 'importlib', 'nn', 'os', 'settings', 'utils']


In [ ]:
"""
2. หลังจากรันครบทุกโมเดลแล้ว
นำ metrics.csv, predict_log.xlsx, confusion_matrix.png ของแต่ละโฟลเดอร์มาเปรียบเทียบ

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ชื่อโฟลเดอร์โมเดล
model_dirs = ['yolov8slr1e-4', 'yolov8s(lre-3)(BG)']
metrics_to_compare = ['mAP50-95', 'mAP50', 'Precision', 'Recall']

summary = []

for d in model_dirs:
    metrics_path = os.path.join(d, 'metrics.csv')
    if not os.path.exists(metrics_path):
        print(f"⚠️ ไม่พบไฟล์: {metrics_path}")
        continue

    df = pd.read_csv(metrics_path)

    row = {'Model': d}
    for m in metrics_to_compare:
        try:
            val = df.loc[df['Metric'] == m, 'Overall'].values
            row[m] = float(val[0]) if len(val) > 0 else np.nan
        except:
            row[m] = np.nan

    summary.append(row)

df_compare = pd.DataFrame(summary)

print(df_compare)

# -----------------------------
# สร้างกราฟเปรียบเทียบ
# -----------------------------

x = np.arange(len(metrics_to_compare))  # ตำแหน่งของ metric
width = 0.35                            # ความกว้างแท่ง

plt.figure(figsize=(10,6))

bars1 = plt.bar(x - width/2, df_compare.iloc[0][metrics_to_compare], width, label=df_compare.iloc[0]['Model'])
bars2 = plt.bar(x + width/2, df_compare.iloc[1][metrics_to_compare], width, label=df_compare.iloc[1]['Model'])

# ใส่ตัวเลขบนแท่ง
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2,
                 height,
                 f'{height:.3f}',
                 ha='center', va='bottom', fontsize=9)

plt.xticks(x, metrics_to_compare)
plt.ylim(0,1)
plt.ylabel('Score')
plt.title('Comparison of Two YOLO Models')
plt.legend()
plt.tight_layout()
plt.show()


⚠️ ไม่พบไฟล์: yolov8slr1e-4\metrics.csv
                Model  mAP50-95     mAP50  Precision    Recall
0  yolov8s(lre-3)(BG)  0.944397  0.975105   0.974344  0.951225


IndexError: single positional indexer is out-of-bounds

In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# ใส่ path ของ results.csv
# =========================
model_results = {
    "YOLOv8s_lr1e-4": r"C:\train_5_2_69\yolov8slre-4.csv",
    "YOLOv8s_lr1e-3_BG": r"C:\train_5_2_69\runs\detect\yolov8s(lre-3)(BG)\train4\results.csv",
}

# โฟลเดอร์เก็บกราฟ
save_dir = r"C:\train_5_2_69\comparison_plots"
os.makedirs(save_dir, exist_ok=True)

# =========================
# โหลดข้อมูล
# =========================
loaded = {}

for name, path in model_results.items():
    if not os.path.exists(path):
        print(f"⚠️ ไม่พบไฟล์: {path}")
        continue
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    loaded[name] = df

if len(loaded) < 2:
    raise ValueError("ต้องมี results.csv อย่างน้อย 2 ไฟล์")

# หา column epoch
def get_epoch_col(df):
    for c in ["epoch", "Epoch"]:
        if c in df.columns:
            return c
    return None

# =========================
# วนลูปสร้างกราฟทุก metric
# =========================
for metric in loaded[list(loaded.keys())[0]].columns:

    if metric.lower() in ["epoch"]:
        continue

    plt.figure(figsize=(10,5))
    plotted = False

    for model_name, df in loaded.items():
        epoch_col = get_epoch_col(df)
        x = df[epoch_col] if epoch_col else df.index

        if metric in df.columns:
            plt.plot(x, df[metric], label=model_name)
            plotted = True

    if not plotted:
        plt.close()
        continue

    plt.title(f"Comparison: {metric}")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.tight_layout()

    # ตั้งชื่อไฟล์ปลอดภัย
    safe_metric = metric.replace("/", "_").replace("(", "").replace(")", "")
    save_path = os.path.join(save_dir, f"{safe_metric}.png")

    plt.savefig(save_path, dpi=300)
    plt.close()

print(f"✅ บันทึกกราฟทั้งหมดไว้ที่: {save_dir}")


✅ บันทึกกราฟทั้งหมดไว้ที่: C:\train_5_2_69\comparison_plots


In [12]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG (FIXED PATH)
# =========================
model_dirs = [
    r'C:\train20_12_68\results_yolov8n\train',
    r'C:\train20_12_68\results_yolov8s\train',
    r'C:\train20_12_68\results_yolov8s(รอบ2)\train',
    r'C:\train20_12_68\results_yolov8s(lr1e-4)(1)\train',
]

OUTPUT_DIR = r'C:\train20_12_68\plots'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# COLUMNS
# =========================
metrics_cols = {
    'metrics/precision(B)': 'Precision',
    'metrics/recall(B)': 'Recall',
    'metrics/mAP50(B)': 'mAP50',
    'metrics/mAP50-95(B)': 'mAP50-95'
}

loss_cols = {
    'train/box_loss': 'Train Box Loss',
    'val/box_loss': 'Val Box Loss',
    'train/cls_loss': 'Train Cls Loss',
    'val/cls_loss': 'Val Cls Loss',
    'train/dfl_loss': 'Train DFL Loss',
    'val/dfl_loss': 'Val DFL Loss',
}

# =========================
# MAIN
# =========================
for model_path in model_dirs:
    results_path = os.path.join(model_path, 'results.csv')

    if not os.path.exists(results_path):
        print(f'Skip {model_path}')
        continue

    df = pd.read_csv(results_path)
    model_name = os.path.basename(os.path.dirname(model_path))

    # ----- Metrics -----
    plt.figure(figsize=(10,6))
    for col, label in metrics_cols.items():
        plt.plot(df['epoch'], df[col], label=label)

    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.title(f'YOLO Metrics vs Epoch - {model_name}')
    plt.legend()
    plt.grid(True)
    plt.ylim(0, 1)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{model_name}_metrics.png'), dpi=300)
    plt.close()

    # ----- Loss -----
    plt.figure(figsize=(10,6))
    for col, label in loss_cols.items():
        plt.plot(df['epoch'], df[col], label=label)

    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'YOLO Train vs Val Loss - {model_name}')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{model_name}_loss.png'), dpi=300)
    plt.close()

    print(f'✔ Finished: {model_name}')

print('🎉 All plots saved successfully')


✔ Finished: results_yolov8n
✔ Finished: results_yolov8s


C:\Users\User\AppData\Local\Temp\ipykernel_15100\1574593672.py:62: UserWarning: Glyph 3619 (\N{THAI CHARACTER RO RUA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\User\AppData\Local\Temp\ipykernel_15100\1574593672.py:62: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\User\AppData\Local\Temp\ipykernel_15100\1574593672.py:62: UserWarning: Glyph 3610 (\N{THAI CHARACTER BO BAIMAI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\User\AppData\Local\Temp\ipykernel_15100\1574593672.py:63: UserWarning: Glyph 3619 (\N{THAI CHARACTER RO RUA}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, f'{model_name}_metrics.png'), dpi=300)
C:\Users\User\AppData\Local\Temp\ipykernel_15100\1574593672.py:63: UserWarning: Glyph 3629 (\N{THAI CHARACTER O ANG}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, f'{model_name}_metrics.png'), dpi=300)
C:\Users\User\A

✔ Finished: results_yolov8s(รอบ2)
✔ Finished: results_yolov8s(lr1e-4)(1)
🎉 All plots saved successfully
